# VLM Mini Workshop: Image + Text in 1 Hour

This notebook is designed for non-technical learners.

## What you will learn
1. What a Vision-Language Model (VLM) is
2. How a model can caption an image (image -> text)
3. How a model can answer questions about an image (image + question -> answer)

## Workshop timing (approx.)
- Setup and dataset: 10-15 min
- Caption demo: 15 min
- Visual Q&A demo: 15 min
- Hands-on + discussion: 15 min


## Cell 1 - One-time Colab setup

In Google Colab:
- Runtime -> Change runtime type -> T4 GPU
- Open the key icon (Secrets) and add:
  - KAGGLE_USERNAME
  - KAGGLE_KEY

Get these from https://www.kaggle.com/settings/account


In [ ]:
import platform
import sys
import torch

print('Python:', sys.version.split()[0])
print('OS:', platform.platform())
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Cell 2 - Install minimal packages


In [ ]:
!pip -q install --upgrade pip
!pip -q install kaggle transformers pillow matplotlib
print('Packages installed.')


## Cell 3 - Download a small public Kaggle dataset

Dataset used: alxmamaev/flowers-recognition

To keep things fast, we sample only a few images per class.


In [ ]:
import glob
import json
import os
import random
import shutil
from pathlib import Path

from google.colab import userdata

username = userdata.get('KAGGLE_USERNAME').strip()
key = userdata.get('KAGGLE_KEY').strip()

if key.startswith('{'):
    parsed = json.loads(key)
    username = parsed.get('username', username).strip()
    key = parsed.get('key', key).strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w', encoding='utf-8') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

DATA_ROOT = Path('/content/vlm_workshop_data')
RAW_DIR = DATA_ROOT / 'raw'
SAMPLE_DIR = DATA_ROOT / 'sample_images'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

print('Downloading from Kaggle...')
!kaggle datasets download -d alxmamaev/flowers-recognition -p {RAW_DIR} --force
!unzip -q -o {RAW_DIR}/flowers-recognition.zip -d {RAW_DIR}

flowers_root_candidates = [RAW_DIR / 'flowers', RAW_DIR / 'flowers' / 'flowers']
flowers_root = next((p for p in flowers_root_candidates if p.exists()), None)
if flowers_root is None:
    raise FileNotFoundError('Could not find extracted flower folder.')

classes = sorted([p.name for p in flowers_root.iterdir() if p.is_dir()])
print('Classes:', classes)

random.seed(42)
max_per_class = 4
for class_name in classes:
    src_class = flowers_root / class_name
    dst_class = SAMPLE_DIR / class_name
    os.makedirs(dst_class, exist_ok=True)

    imgs = sorted(glob.glob(str(src_class / '*.jpg'))) + sorted(glob.glob(str(src_class / '*.jpeg'))) + sorted(glob.glob(str(src_class / '*.png')))
    random.shuffle(imgs)
    imgs = imgs[:max_per_class]

    for img in imgs:
        shutil.copy(img, dst_class / Path(img).name)

all_sample_images = sorted(SAMPLE_DIR.glob('*/*'))
print('Small sample created at:', SAMPLE_DIR)
print('Total sampled images:', len(all_sample_images))


## Cell 4 - Quick image preview


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

preview_images = all_sample_images[:8]
plt.figure(figsize=(14, 7))
for i, path in enumerate(preview_images, start=1):
    plt.subplot(2, 4, i)
    plt.imshow(Image.open(path).convert('RGB'))
    plt.title(path.parent.name)
    plt.axis('off')
plt.tight_layout()
plt.show()


## Cell 5 - Load caption model (image -> text)

Model: Salesforce/blip-image-captioning-base


In [ ]:
from transformers import BlipForConditionalGeneration, BlipProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
caption_model_id = 'Salesforce/blip-image-captioning-base'

caption_processor = BlipProcessor.from_pretrained(caption_model_id)
caption_model = BlipForConditionalGeneration.from_pretrained(caption_model_id).to(device)

print('Caption model loaded on:', device)


In [ ]:
def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = caption_processor(images=image, return_tensors='pt').to(device)
    with torch.inference_mode():
        output = caption_model.generate(**inputs, max_new_tokens=30)
    return caption_processor.decode(output[0], skip_special_tokens=True)

for p in all_sample_images[:5]:
    cap = generate_caption(p)
    print(f'Image: {p.name} | Class folder: {p.parent.name}')
    print('Caption:', cap)
    print('-' * 70)


## Cell 6 - Load visual question-answering model

Model: Salesforce/blip-vqa-base

This model takes an image + a text question.


In [ ]:
from transformers import BlipForQuestionAnswering

vqa_model_id = 'Salesforce/blip-vqa-base'
vqa_processor = BlipProcessor.from_pretrained(vqa_model_id)
vqa_model = BlipForQuestionAnswering.from_pretrained(vqa_model_id).to(device)

print('VQA model loaded on:', device)


In [ ]:
def ask_image_question(image_path, question):
    image = Image.open(image_path).convert('RGB')
    inputs = vqa_processor(image, question, return_tensors='pt').to(device)
    with torch.inference_mode():
        output = vqa_model.generate(**inputs, max_new_tokens=20)
    return vqa_processor.decode(output[0], skip_special_tokens=True)

test_image = all_sample_images[0]
print('Test image:', test_image)

questions = [
    'What is in this image?',
    'What color is the flower?',
    'Is this a close-up photo?'
]

for q in questions:
    a = ask_image_question(test_image, q)
    print('Q:', q)
    print('A:', a)
    print()


## Cell 7 - Student mini activity

Try this with your own chosen sample image:
1. Change image index
2. Generate a caption
3. Ask two custom questions
4. Compare answers with your expectation


In [ ]:
idx = 3
student_image = all_sample_images[idx]
print('Selected image:', student_image)
print('Caption:', generate_caption(student_image))
print()

student_questions = [
    'What flower do you think this is?',
    'What is the main visible color?'
]

for q in student_questions:
    print('Q:', q)
    print('A:', ask_image_question(student_image, q))
    print('-' * 60)


## Cell 8 - Save sample outputs (optional)


In [ ]:
results = []
for p in all_sample_images[:10]:
    results.append({
        'image': str(p),
        'class_folder': p.parent.name,
        'caption': generate_caption(p),
        'question': 'What is in this image?',
        'answer': ask_image_question(p, 'What is in this image?')
    })

out_path = '/content/vlm_workshop_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('Saved to:', out_path)
print('Records:', len(results))
